# Imports

In [1]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os
from pathlib import Path

from typing import List, Union

import torch #PyTorch

from tqdm.notebook import tqdm

import cv2 #OpenCV

# Setup

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Config
# Path to dataset (Each group member may need to modify this)
DATASET_RAW_FILEPATH_IN_DRIVE = "/content/drive/MyDrive/COMP-SCI 5530 - Principles of Data Science/Project/data_raw/deepfake-and-real-images.zip"

In [4]:
# Copy data clean zip from Google Drive
!cp "{DATASET_RAW_FILEPATH_IN_DRIVE}" "/content/data_raw.zip"

In [5]:
# Unzip data raw zip
!unzip "/content/data_raw.zip"

Streaming output truncated to the last 5000 lines.
  inflating: Dataset/Validation/Real/real_5499.jpg  
  inflating: Dataset/Validation/Real/real_55.jpg  
  inflating: Dataset/Validation/Real/real_550.jpg  
  inflating: Dataset/Validation/Real/real_5500.jpg  
  inflating: Dataset/Validation/Real/real_5501.jpg  
  inflating: Dataset/Validation/Real/real_5502.jpg  
  inflating: Dataset/Validation/Real/real_5503.jpg  
  inflating: Dataset/Validation/Real/real_5504.jpg  
  inflating: Dataset/Validation/Real/real_5505.jpg  
  inflating: Dataset/Validation/Real/real_5506.jpg  
  inflating: Dataset/Validation/Real/real_5507.jpg  
  inflating: Dataset/Validation/Real/real_5508.jpg  
  inflating: Dataset/Validation/Real/real_5509.jpg  
  inflating: Dataset/Validation/Real/real_551.jpg  
  inflating: Dataset/Validation/Real/real_5510.jpg  
  inflating: Dataset/Validation/Real/real_5511.jpg  
  inflating: Dataset/Validation/Real/real_5512.jpg  
  inflating: Dataset/Validation/Real/real_5513.jpg  

In [6]:
# Print GPU info
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: False
GPU: None


In [7]:
# Attempt to run on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Running on:", device)

Running on: cpu


# Helper Fuctions

In [8]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_faces(img_path: str) -> List[np.ndarray]:
    image = cv2.imread(img_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces_bboxes = face_cascade.detectMultiScale(gray, 1.3, 5)

    face_images = []
    for (x, y, w, h) in faces_bboxes:
        cropped_face = image[y:y+h, x:x+w]
        face_images.append(cropped_face)

    return face_images

# Cleaning Data

In [9]:
raw_dir_root = "/content/Dataset"
clean_dir_root = "/content/data_clean"

# Get total num files in raw data
total_files = 0
for root, dirs, files in os.walk(raw_dir_root):
  total_files += len(files)

with tqdm(total = total_files, desc="Cleaning Data") as pbar:
  for folder in os.listdir(raw_dir_root):
    folder_path = os.path.join(raw_dir_root, folder)
    for subfolder in os.listdir(folder_path):
      subfolder_path = os.path.join(folder_path, subfolder)
      save_path = f"{clean_dir_root}/{folder}/{subfolder}"
      os.makedirs(save_path, exist_ok=True)
      for img_name in os.listdir(subfolder_path):
        img_path = os.path.join(subfolder_path, img_name)
        img_name_parts = img_name.split('.')

        faces = get_faces(img_path=img_path)
        for i, face in enumerate(faces):
          img_name_with_number = f"{img_name_parts[0]}_{i}.{img_name_parts[1]}"
          clean_img_path = os.path.join(save_path, img_name_with_number)
          cv2.imwrite(clean_img_path, face)

        pbar.update(1)

Cleaning Data:   0%|          | 0/190335 [00:00<?, ?it/s]

In [10]:
!zip -r "data_clean.zip" "data_clean"

Streaming output truncated to the last 5000 lines.
  adding: data_clean/Validation/Fake/fake_6443_0.jpg (deflated 2%)
  adding: data_clean/Validation/Fake/fake_13798_0.jpg (deflated 3%)
  adding: data_clean/Validation/Fake/fake_347_0.jpg (deflated 2%)
  adding: data_clean/Validation/Fake/fake_8552_0.jpg (deflated 2%)
  adding: data_clean/Validation/Fake/fake_12024_0.jpg (deflated 1%)
  adding: data_clean/Validation/Fake/fake_15379_0.jpg (deflated 4%)
  adding: data_clean/Validation/Fake/fake_12265_0.jpg (deflated 2%)
  adding: data_clean/Validation/Fake/fake_8371_1.jpg (deflated 4%)
  adding: data_clean/Validation/Fake/fake_4466_0.jpg (deflated 3%)
  adding: data_clean/Validation/Fake/fake_16093_0.jpg (deflated 3%)
  adding: data_clean/Validation/Fake/fake_6791_0.jpg (deflated 2%)
  adding: data_clean/Validation/Fake/fake_17761_0.jpg (deflated 1%)
  adding: data_clean/Validation/Fake/fake_14857_0.jpg (deflated 2%)
  adding: data_clean/Validation/Fake/fake_3477_0.jpg (deflated 2%)
  add

In [11]:
!cp "/content/data_clean.zip" "/content/drive/MyDrive"

In [12]:
from google.colab import runtime
runtime.unassign()